<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/02_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 02 — Tools: Four Flavors

> **⚡ Quick path** — this is one of four modules on the 1-hour course preview.
> See the [README's Quick path section](../README.md#-quick-path---1-hour) for the sequence.

> **Where you are** — after M01 you can build an agent whose tool is a Python function,
> and you know the docstring is the schema.
> - **You can already:** write one tool = typed function + docstring; read the event stream.
> - **New here:** three more *sources* of tools — a REST API, an MCP server, another agent —
>   plus a safety pattern for dangerous tools.
> - **First met here:** a subprocess (a second program your notebook starts) — explained when
>   we get to MCP.

## One Story, Four Flavors

This module has a single story: **you are building the agent for an IT help desk.** Employees ask it things all day — *is the VPN down? what's the status of my ticket? what's this invoice in euros? can you translate this reply?* Each request needs a different kind of tool, and that is exactly ADK's menu of **four tool flavors**:

1. **A function you write yourself** — check whether a company system is up.
2. **Someone else's web API** — live exchange rates for the finance team.
3. **An MCP server** — the ticket database, running as its own program.
4. **Another agent** — a translation specialist the desk can consult.

Plus one closing question: what to do about tools that can *destroy* things.

**Cost:** under $0.02 on OpenRouter. &nbsp; **Colab note:** the MCP demo needs this repo's `mcp_servers/` folder — the setup cell below clones it for you.

# Setup

Same ritual as M01 — install, clone the repo (Colab only), load the key — plus imports with two odd-looking lines explained below.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 mcp==1.29.0 httpx==0.28.1 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


If you're on Colab, the next cell clones the repo so `mcp_servers/` is available; locally it does nothing.

In [2]:
import os
if "COLAB_GPU" in os.environ or "COLAB_JUPYTER_IP" in os.environ:
    !git clone --depth=1 -q https://github.com/robertbarcik/ADK-tutorial /content/ADK-tutorial 2>/dev/null || true
    os.chdir("/content/ADK-tutorial")
    print(f"✅ Colab: working in {os.getcwd()}")
else:
    # Local: you should already be inside the repo.
    print(f"✅ Local: working in {os.getcwd()}")

✅ Local: working in /Users/robertbarcik/git-repos/ADK-tutorial/notebooks


Same OpenRouter key as M01 — picked up automatically from Colab secrets or `.env`.

In [3]:
import os

OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Tip: set OPENROUTER_API_KEY in Colab secrets (🔑 icon) or a local .env file.")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


## Imports

The imports that matter are the flavors themselves: `OpenAPIToolset`, `McpToolset`, `AgentTool` — and `FunctionTool`, which needs no explicit import (you pass the bare function).

The cell also starts with two blocks of **notebook plumbing** — small compatibility patches Jupyter needs before a notebook can launch and talk to a second program in the background (Flavor 3 will do exactly that). Run them and move on; there is nothing to learn in them.

*Optional detail, only if you're curious:* `nest_asyncio` — Jupyter already runs an async event loop and the MCP client wants to start its own; the patch lets one nest inside the other. The `sys.stderr` swap gives the MCP subprocess launcher a real file handle for error output, which Jupyter's replacement `stderr` doesn't provide.

In [4]:
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")

# Jupyter / Colab patch sys.stderr with an OutStream that lacks a usable .fileno().
# MCP's stdio subprocess spawn needs fileno(), so we swap stderr to a real fd.
# stdout is left alone, so print() still shows normally in the notebook.
try:
    sys.stderr.fileno()
except Exception:
    import os
    sys.stderr = open(os.devnull, "w")

import nest_asyncio
nest_asyncio.apply()

os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm
litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.openapi_tool import OpenAPIToolset
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from google.adk.tools.agent_tool import AgentTool
from mcp import StdioServerParameters
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


## The `chat()` Helper from M01

Same helper as in M01 (session → runner → loop over events), with two small additions: an optional app name, and truncation of long outputs — MCP tool responses can be hundreds of characters of JSON. If `async def` / `await` still feels foreign, re-read M01's two rules; nothing new happens here.

In [5]:
APP = "m02_tools"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str, app: str = APP):
    sid = f"s-{uuid.uuid4().hex[:8]}"
    await session_service.create_session(app_name=app, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=app, session_service=session_service)
    message = types.Content(role="user", parts=[types.Part(text=prompt)])

    print(f"USER: {prompt}\n")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        tag = "[FINAL]" if event.is_final_response() else "[step]"
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    print(f"{tag} {event.author}: {p.text.strip()[:400]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"[tool_call] {p.function_call.name}({args})")
                if p.function_response:
                    resp = str(p.function_response.response)
                    # Truncate long MCP responses
                    print(f"[tool_resp] {resp[:350]}{'...' if len(resp) > 350 else ''}")

print("✅ chat() helper ready.")

✅ chat() helper ready.


# How Every Tool Works (Regardless of Flavor)

```
┌─────────────┐   "I need to know X"    ┌─────────────┐
│   LLM       │────────────────────────▶│    ADK      │
│   decides   │                         │  dispatch   │
└─────────────┘                         └──────┬──────┘
       ▲                                       │
       │  "X is {...}"                         ▼
       │                                 ┌─────────────┐
       └─────────────────────────────────│  your tool  │
                                         │  (any kind) │
                                         └─────────────┘
```

The model sees a **schema** (name, description, argument types) and decides when to call the tool. ADK does the dispatch: it finds the code behind the schema, runs it, and hands the result back to the model as a tool-response event. You built exactly this pipeline by hand in the previous course — schema, dispatch dict, loop.

The four flavors differ only in **where the schema comes from** and **where the code lives**:

| # | Flavor | Schema comes from | Code lives in |
|---|---|---|---|
| 1 | **FunctionTool** | docstring + type hints | your notebook |
| 2 | **OpenAPIToolset** | an OpenAPI spec | a remote HTTP API |
| 3 | **McpToolset** | the server's `list_tools` answer | a separate process |
| 4 | **AgentTool** | the agent's `name` + `description` | that other agent |

This table returns at the start of every flavor, so you always know where you are.

# Flavor 1 of 4 — A Function You Write Yourself

| # | Flavor | |
|---|---|---|
| 1 | A function you write yourself | **→ now** |
| 2 | Someone else's web API (OpenAPI) |  |
| 3 | An MCP server |  |
| 4 | Another agent as a tool |  |

First request of the day: *"I can't connect to the VPN — is it down?"* The help desk needs to check system status, and no public API knows *your company's* systems — so you write the function yourself. This is the flavor you know from M01; here is the checklist that makes such functions **good** tools:

1. **Docstring for the model, not for a reviewer** — the model has no Slack channel to ask you what a parameter means.
2. **Type hints matter here** — `system: str` becomes `"type": "string"` in the schema; without them the model guesses.
3. **Return a JSON-serializable dict or string** — the return value goes verbatim to the model.

One new Python bit in the code: `detail: str = "basic"` — a parameter with a **default value**. Python treats it as optional, and ADK marks it optional in the schema, so the model may leave it out.

In [6]:
def check_system_status(system: str, detail: str = "basic") -> dict:
    """Check whether a company IT system is currently up.

    Use this tool whenever an employee asks if something is down, slow, or
    unreachable. Known systems: 'vpn', 'mail', 'printers', 'wiki'.

    Args:
        system: Which system to check — one of 'vpn', 'mail', 'printers', 'wiki'.
        detail: 'basic' (default) returns the status only; 'full' adds uptime
            and the last incident.
    """
    fake_status = {
        "vpn":      ("degraded", "99.1%", "packet loss since 09:40"),
        "mail":     ("up",       "99.9%", "none this month"),
        "printers": ("down",     "97.2%", "toner outage, 3rd floor"),
        "wiki":     ("up",       "99.8%", "none this month"),
    }
    if system not in fake_status:
        return {"system": system, "error": f"Unknown system '{system}'. Known: {sorted(fake_status)}"}
    status, uptime, incident = fake_status[system]
    result = {"system": system, "status": status}
    if detail == "full":
        result.update({"uptime_30d": uptime, "last_incident": incident})
    return result

helpdesk_agent = LlmAgent(
    name="helpdesk_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="First-line IT help desk: checks whether company systems are up.",
    instruction=(
        "You are an IT help desk assistant. When an employee reports a problem "
        "with a system, call check_system_status and answer from its result. "
        "Be brief and factual."
    ),
    tools=[check_system_status],
)

await chat(helpdesk_agent, "I can't connect to the VPN — is it down?")

USER: I can't connect to the VPN — is it down?



[tool_call] check_system_status({'system': 'vpn', 'detail': 'basic'})
[tool_resp] {'system': 'vpn', 'status': 'degraded'}


[FINAL] helpdesk_agent: The VPN is currently degraded, so connectivity issues may occur.


### 🔍 What just happened?

- The model mapped "can't connect to the VPN" to `system='vpn'` — the docstring's list of known systems did the same job your hand-written `enum` used to do.
- It passed `detail='basic'` explicitly this run — but it didn't have to: the default value made that argument optional (the schema print below shows it outside `required`).
- The tool returned a **dict**, not prose; the model composed the sentence around the structured data.

FunctionTool is the default flavor. Reach for it unless one of the next three clearly fits better.

In [7]:
from google.adk.tools.function_tool import FunctionTool
import json as _json

# The same peek as in M01: what does the model actually receive?
decl = FunctionTool(check_system_status)._get_declaration()
print("name:       ", decl.name)
print("description:", (decl.description or "").split("\n")[0], "...")
print(_json.dumps(decl.parameters_json_schema, indent=2))
print()
print("💡 Note 'required': only 'system' is listed — the default value made 'detail' optional.")

name:        check_system_status
description: Check whether a company IT system is currently up. ...
{
  "properties": {
    "system": {
      "title": "System",
      "type": "string"
    },
    "detail": {
      "default": "basic",
      "title": "Detail",
      "type": "string"
    }
  },
  "required": [
    "system"
  ],
  "title": "check_system_statusParams",
  "type": "object"
}

💡 Note 'required': only 'system' is listed — the default value made 'detail' optional.


### 🎯 Mini-task

Ask the agent *"Is the mail server OK? Give me the full details."* — does the model pass `detail='full'`? Then ask about *"the coffee machine"* and watch how it handles the error dict.

# Flavor 2 of 4 — Someone Else's Web API

| # | Flavor | |
|---|---|---|
| 1 | A function you write yourself | ✅ |
| 2 | Someone else's web API (OpenAPI) | **→ now** |
| 3 | An MCP server |  |
| 4 | Another agent as a tool |  |

Next request, this time from finance: *"An invoice came in Swiss francs — what is that in euros today?"*

Our status tool worked because we wrote the Python ourselves. But exchange rates don't live in your notebook — they live in **someone else's web service**. You *could* write a `FunctionTool` that calls it with `requests.get(...)`, then another one for the next endpoint, and another… Or you hand ADK the API's **menu** and get all of its operations as tools at once.

That menu is called an **OpenAPI spec** — the same "menu" idea as the tool schema you hand-wrote last course, just for a *whole API*: for every URL path, which method, which parameters, what comes back. Many APIs publish theirs at `/openapi.json`, and frameworks like FastAPI generate it for free.

The demo uses [Frankfurter](https://api.frankfurter.dev) — a free, no-auth currency-rates API. We feed ADK a minimal hand-written slice of its spec. Read it once, top to bottom: *servers* (base URL) → *paths* (`/latest`) → *get* → *parameters* (`base`, `symbols`) → *responses*. And look at the `description` strings especially — they play exactly the role the docstring played in Flavor 1, because they are what the model sees.

### Where does a spec come from, in real life?

You normally don't write it. The API's makers publish it — as a download on their documentation page, or live at an address like `https://api.example.com/openapi.json`. (APIs built with frameworks like FastAPI generate it automatically.) You fetch it, load it into a dict, hand it to ADK. Ours is hand-written only so it fits on one screen.

And what does "calling the API" physically look like? One HTTP GET — a URL with the arguments packed in after the `?`:

```
https://api.frankfurter.dev/v1/latest?base=CHF&symbols=EUR
└─────────── server ─────────┘└─path┘└──── arguments ────┘
```

Paste that into a browser and you'll see the JSON answer. That whole URL is what ADK assembles — from the spec plus the model's arguments — every time the model calls this tool.

### The key line, before you run it

```python
fx_toolset = OpenAPIToolset(spec_dict=FRANKFURTER_SPEC)
```

Conceptually, nothing new happens here. In Flavor 1, ADK read your *function* and made a tool out of it: schema from the docstring, code = your Python. This line does the same reading from a different source: it reads the *spec* and makes one tool per operation — schema from the spec's `description` fields, and the "code" is an HTTP request that ADK writes and sends for you. Same schema + code pair as always; only where each comes from changed, because this time the code runs on someone else's server.

One more small thing in the cell below: `tools=[fx_toolset]` instead of `tools=[some_function]`. A **toolset** is simply a bundle of tools — you drop the whole bundle into the same slot.

In [8]:
FRANKFURTER_SPEC = {
    "openapi": "3.0.0",
    "info": {"title": "Frankfurter FX", "version": "1.0.0"},
    "servers": [{"url": "https://api.frankfurter.dev/v1"}],
    "paths": {
        "/latest": {
            "get": {
                "operationId": "getLatestRate",
                "summary": "Get today's exchange rate between two currencies.",
                "parameters": [
                    {
                        "name": "base",
                        "in": "query",
                        "description": "Source currency as a 3-letter ISO code (e.g. USD, EUR, CHF).",
                        "required": False,
                        "schema": {"type": "string"},
                    },
                    {
                        "name": "symbols",
                        "in": "query",
                        "description": "Comma-separated target currency codes (e.g. 'EUR' or 'EUR,GBP').",
                        "required": False,
                        "schema": {"type": "string"},
                    },
                ],
                "responses": {
                    "200": {
                        "description": "JSON object with 'base', 'date', and 'rates' fields.",
                        "content": {"application/json": {"schema": {"type": "object"}}},
                    }
                },
            }
        }
    },
}

fx_toolset = OpenAPIToolset(spec_dict=FRANKFURTER_SPEC)

fx_agent = LlmAgent(
    name="fx_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports live foreign exchange rates.",
    instruction=(
        "You are a currency assistant. For a request like 'USD to EUR', call "
        "get_latest_rate with base='USD' and symbols='EUR'. Currency codes are "
        "3-letter ISO codes (USD, EUR, GBP, CHF, JPY). Report the numeric rate."
    ),
    tools=[fx_toolset],
)

await chat(fx_agent, "What's the current rate from CHF to JPY?")

USER: What's the current rate from CHF to JPY?



[tool_call] get_latest_rate({'base': 'CHF', 'symbols': 'JPY'})
[tool_resp] {'amount': 1.0, 'base': 'CHF', 'date': '2026-08-24', 'rates': {'JPY': 198.25}}


[FINAL] fx_agent: 1 CHF = 198.25 JPY.


### 🔍 What just happened?

- The event stream shows `get_latest_rate` — **not** `getLatestRate`. ⚠️ ADK snake-cases operation IDs from specs; if a tool mysteriously "isn't being called", check whether it got renamed.
- The tool-response is the raw HTTP JSON body. ADK doesn't transform the API's output — it ferries it to the model, which picked out the rate itself. Handy side effect: you debug the API's real contract right in the event stream.

In production you'd point this at your company's existing API spec — the one your web frontend already uses — and get all its endpoints as agent tools in one line.

### 🎯 Mini-task

Add a second path to `FRANKFURTER_SPEC` — `/currencies`, a GET that returns all supported currency codes — and ask the agent *"What currencies do you support?"*. Does it call the new operation?

# Flavor 3 of 4 — An MCP Server

| # | Flavor | |
|---|---|---|
| 1 | A function you write yourself | ✅ |
| 2 | Someone else's web API (OpenAPI) | ✅ |
| 3 | An MCP server | **→ now** |
| 4 | Another agent as a tool |  |

Third request: *"What's the status of my ticket about the WiFi?"* The tickets live in a **ticket database**, and this time the tools already exist — this repo ships a small **MCP server** that exposes them.

*MCP server* sounds like heavy machinery. It isn't — and to prove it, we open the box **before** any protocol talk.

### First, Look Inside — It's Just Functions

The next cell imports `ticket_mcp_server.py` **as a plain Python module** and pokes at it directly. No MCP anywhere yet:

In [9]:
import sys
from pathlib import Path

# Make mcp_servers/ importable (repo root, notebooks/, or Colab).
for parent in [Path.cwd(), Path.cwd().parent, Path("/content/ADK-tutorial")]:
    if (parent / "mcp_servers").exists():
        sys.path.insert(0, str(parent / "mcp_servers"))
        break

import ticket_mcp_server as ticket_server

# 1. The "database" is a plain dict — same idea as FAKE_WEATHER in M01:
print("Tickets in the db:", list(ticket_server.TICKETS), "\n")

# 2. And the server's tools are ordinary Python you can call directly, right now:
result = await ticket_server.call_tool("search_tickets", {"query": "wifi"})
print(result[0].text[:400])

Tickets in the db: ['T-1001', 'T-1002', 'T-1003', 'T-1004'] 

{
  "query": "wifi",
  "count": 1,
  "tickets": [
    {
      "id": "T-1003",
      "title": "WiFi connectivity issues",
      "description": "WiFi keeps disconnecting every 10 minutes in Conference Room B",
      "status": "in_progress",
      "priority": "high",
      "assigned_to": "network_team",
      "created_at": "2025-10-05T09:15:00",
      "updated_at": "2025-10-05T11:30:00"
    }
  ]
}


### 🔍 Nothing magical inside

You just called the server's function **directly** — a dict lookup and a loop, nothing more. And here's the best part: open `mcp_servers/ticket_mcp_server.py` and you will recognise *your own pattern from the previous course* — `list_tools()` returns hand-written schemas (yes, the 34-line kind!), and `call_tool(name, arguments)` is a big if/elif dispatch: your `available_functions` idea, written out as a program.

**So what does MCP add? Only the packaging.** It takes functions like these and wraps them as a small stand-alone program — a *tool server*. Any agent that speaks the protocol — ADK, Claude, anything — can then start a conversation with that program and ask it two things: *"what tools do you have?"* and *"run this one for me."* That's the whole idea. What you gain: the tools now live in their own program — it can be written in any language, keep its own data, and serve many different agents.

Same schema + code pair as every flavor, one more layer of wrapping. With that picture in place, the recap:

## MCP in 60 Seconds — a Recap

You have met MCP twice before, from two different angles:

- **Previous course (hosted tools notebook):** you used a *remote* MCP server as an OpenAI hosted
  tool — `{"type": "mcp", "server_label": "gitmcp", "server_url": "https://..."}`. You never saw
  the server or the client; OpenAI ran the MCP client for you, server-side, and tool calls just
  showed up in your response.
- **MCP course (if you took it):** you built the other side yourself — stdio servers like
  `ticket_server.py` in the five-server IT-support system, with `list_tools` / `call_tool`
  handlers. This repo's `ticket_mcp_server.py` is the same idea, trimmed for this demo.

If you skipped the MCP course or it's been a while, everything you need for this module fits in
one sentence: **an MCP server is a separate program that exposes tools; a client connects, asks
`list_tools`, and then calls `call_tool` — the "universal adapter" between agents and tools.**


## What's Different This Time

| | Server runs | MCP client is | You see |
|---|---|---|---|
| Previous course (hosted tool) | remote, someone else's | OpenAI's, server-side | only the results |
| MCP course | your machine | your own code | both sides, by hand |
| **This module** | your machine, **launched by ADK as a subprocess** | **ADK** (inside `McpToolset`) | the handshake, up close |

A *subprocess* just means: your notebook starts a second program (here, a second Python running
`ticket_mcp_server.py`) and keeps it running in the background. The two talk over stdin/stdout —
the same channels `print()` and `input()` use. No network, no ports.


## Connecting with `McpToolset`

One institutional note since your last encounter: MCP was donated in December 2025 to the Agentic AI Foundation (a Linux Foundation project co-founded by Anthropic, OpenAI and Block) and is now the de facto standard for agent-to-tool communication. The server can be written in any language and keep its own state — database connections, credentials. Your agent only speaks the protocol.

This repo ships three servers under `mcp_servers/`; we'll use `ticket_mcp_server.py` (five tools: get, list, create, update, search). `McpToolset` launches it as a subprocess and discovers its tools automatically.

The connection line below looks scary — three constructors nested in one line:

```python
McpToolset(connection_params=StdioConnectionParams(server_params=StdioServerParameters(...)))
```

Read it from the inside out; it answers three plain questions:

- *Which program should be started?* — the innermost part. Ours says: "Python, running `ticket_mcp_server.py`". (`sys.executable` just means "the same Python this notebook runs on".)
- *How do we talk to it?* — the middle part. Through the program's ordinary text input and output — the same channels `print()` and `input()` use — with a time limit.
- *Who handles all of that?* — the outermost part hands the bundle to ADK: start the program, ask it *"what tools do you have?"*, and offer whatever comes back to our agent. Exactly what you did by hand a moment ago, automated.

In [10]:
import os
from pathlib import Path

# Resolve mcp_servers/ticket_mcp_server.py whether the notebook runs from the
# repo root (local) or from notebooks/ (nbconvert / Jupyter Lab default cwd).
def _resolve_mcp_server(script_name: str) -> str:
    for parent in [Path.cwd(), Path.cwd().parent, Path("/content/ADK-tutorial")]:
        candidate = parent / "mcp_servers" / script_name
        if candidate.exists():
            return str(candidate.resolve())
    raise FileNotFoundError(f"Could not locate mcp_servers/{script_name}")

TICKET_SERVER_PATH = _resolve_mcp_server("ticket_mcp_server.py")
print(f"✅ MCP server script: {TICKET_SERVER_PATH}")

ticket_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,  # use the same Python that's running this notebook
            args=[TICKET_SERVER_PATH],
        ),
        timeout=20,
    )
)

ticket_agent = LlmAgent(
    name="ticket_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Helps users look up and manage IT support tickets.",
    instruction=(
        "You help the IT support team. Use search_tickets to find tickets by "
        "keyword, get_ticket by ID, list_tickets to browse, update_ticket to "
        "change a status, and create_ticket for new issues. Be concise."
    ),
    tools=[ticket_toolset],
)

await chat(ticket_agent, "Find any open tickets about WiFi.")

✅ MCP server script: /Users/robertbarcik/git-repos/ADK-tutorial/mcp_servers/ticket_mcp_server.py
USER: Find any open tickets about WiFi.



[step] ticket_agent: **Figuring out the list requirements**

I’m thinking we need a list with an open status, and maybe no additional filters are necessary. I see that the schema requires all fields to be strings. It mentions that no fields can be optional, but maybe I can still pass empty values for the priority assigned. I’ll have to double-check that to make sure I’m on the right track. It’s always good to keep thi
[tool_call] list_tickets({'status': 'open', 'priority': 'low', 'assigned_to': ''})
[tool_resp] {'content': [{'type': 'text', 'text': '{\n  "count": 1,\n  "tickets": [\n    {\n      "id": "T-1004",\n      "title": "Software installation request",\n      "description": "Need Adobe Photoshop installed for design work",\n      "status": "open",\n      "priority": "low",\n      "assigned_to": "software_team",\n      "created_at": "2025-10-06T08:0...


[step] ticket_agent: **Searching for tickets**

I need to figure out the right keywords to use for this search. Once I have those, I can filter the results to find open tickets. It seems straightforward enough, but I want to ensure I'm being thorough. I should consider what additional criteria might help narrow it down further as I go along. So, let's start with those keywords and see how the search results look!
[tool_call] search_tickets({'query': 'WiFi'})
[tool_resp] {'content': [{'type': 'text', 'text': '{\n  "query": "WiFi",\n  "count": 1,\n  "tickets": [\n    {\n      "id": "T-1003",\n      "title": "WiFi connectivity issues",\n      "description": "WiFi keeps disconnecting every 10 minutes in Conference Room B",\n      "status": "in_progress",\n      "priority": "high",\n      "assigned_to": "network_team",...


[FINAL] ticket_agent: No open WiFi tickets found. The only WiFi ticket, **T-1003**, is currently **in progress**.


### 🔍 What just happened?

`McpToolset` started the server as a subprocess, spoke the MCP handshake over stdio, got back schemas for five tools, and registered them as if they were local functions — the agent just sees `search_tickets` and calls it.

Two new things in this stream:

- A `[step]` text event *before* the tool calls — GPT-5.6's short **reasoning summary**, the "thinking" you know from *Pod kapotou*, surfacing as ordinary text (other vendors hide it; M04 compares).
- **Two tool calls in one event** — the model asked for `search_tickets` and `list_tickets` at once, and ADK ran both. Parallel tool calls are a model capability, not something you configure.

⚠️ An MCP toolset owns a live subprocess — we close it at the end of the notebook.

### 🎯 Mini-task

Swap `ticket_mcp_server.py` for `knowledge_mcp_server.py` and build an agent that searches help articles. What tools does it discover?

# Flavor 4 of 4 — Another Agent as a Tool

| # | Flavor | |
|---|---|---|
| 1 | A function you write yourself | ✅ |
| 2 | Someone else's web API (OpenAPI) | ✅ |
| 3 | An MCP server | ✅ |
| 4 | Another agent as a tool | **→ now** |

Last request of the day: *"A colleague in Vienna asked for this reply in Slovak — can you translate it?"* Translation isn't a lookup — it needs a language model of its own. So the help desk **consults a specialist**: a second agent, wrapped so the first one can call it *like a function*.

That is `AgentTool`: the specialist answers one question and hands control straight back — the parent never stops being in charge. The key line below is `tools=[AgentTool(agent=translator)]` — the same wrapping move as every flavor: something that can answer (this time a whole agent) gets a schema (from its `name` and `description`) and becomes callable like a function. (The other multi-agent pattern, `sub_agents`, *transfers* the whole conversation to the child; M06 puts the two side by side.)

In [11]:
# A specialist agent that translates short phrases to Slovak.
translator = LlmAgent(
    name="translator",
    model=LiteLlm(model=MODEL_STRING),
    description="Translates a short English phrase into Slovak. Input: the phrase. Output: the Slovak translation.",
    instruction=(
        "You are a translation tool. Input is an English phrase. Output is the "
        "Slovak translation only — no commentary, no prefix, no quotes. Respond "
        "with the translation and nothing else."
    ),
)

# Wrap the translator as a callable tool, then give it to a parent agent.
orchestrator = LlmAgent(
    name="orchestrator",
    model=LiteLlm(model=MODEL_STRING),
    description="A helpful assistant that can translate phrases via a specialist.",
    instruction=(
        "You answer questions. When the user asks for a translation to Slovak, "
        "call the translator tool with the phrase to translate. Include the "
        "translator's result in your reply."
    ),
    tools=[AgentTool(agent=translator)],
)

await chat(orchestrator, "How do you say 'good morning, my friend' in Slovak?")

USER: How do you say 'good morning, my friend' in Slovak?



[tool_call] translator({'request': 'good morning, my friend'})


[tool_resp] {'result': 'Dobré ráno, môj priateľu'}


[FINAL] orchestrator: Dobré ráno, môj priateľu.


### 🔍 What just happened?

The parent called `translator` as if it were a function. The specialist ran in its own fresh LLM call, produced the translation, and the parent wrapped the result into its reply — all of it visible in the parent's event stream.

`AgentTool` is the right choice when the child has a clean input→output contract and the parent should stay in charge. Use `sub_agents` when the child should own the conversation for a stretch of turns — M06 makes the distinction crisp with a side-by-side demo.

### 🎯 Mini-task

Wrap `fx_agent` from Flavor 2 as a second `AgentTool` on the orchestrator and ask: *"Translate 'Today 1 CHF is X EUR' into Slovak, with X filled from the live rate."* Does it call both tools in one turn?

# Which Tools Are Dangerous?

> *From the "Agentic Design Patterns" publication, Pattern 4. Two minutes of theory, then a demo.*

Our help desk can read tickets. Should it be able to **delete** them? Not all tools are equal — the difference is **blast radius**: how much damage a misfiring tool call does before anyone notices.

| Risk tier | Characteristic | Example | Guard |
|---|---|---|---|
| **Read-only** | Doesn't change anything external | `check_system_status`, `search_tickets` | None needed |
| **Mutating, reversible** | Writes, but easy to undo | `create_ticket`, `send_draft_email` | Log every call |
| **Mutating, irreversible** | Writes that can't be rolled back | `charge_credit_card`, `execute_sql` | **Explicit confirmation** |
| **Catastrophic** | Destructive, multi-user, loud | `drop_database`, `delete_user` | **Human in the loop** — never callable directly |

The temptation is to treat every tool the same, because the framework does. Don't. And put the guard **in the tool's code**, not in the instruction — an instruction is a polite request the model can ignore; a code-level check is a wall. Here is the minimal pattern: a delete that refuses to run without a confirmation token.

In [12]:
# Simulated ticket DB so we can "delete" from it.
TICKETS = {"T-1001": "Laptop won't boot", "T-1002": "Password reset request"}

def delete_ticket(ticket_id: str, confirmation_token: str = "") -> dict:
    """Delete a support ticket. IRREVERSIBLE.

    Args:
        ticket_id: The ticket to delete, e.g. "T-1001".
        confirmation_token: MUST equal "CONFIRM_DELETE_<ticket_id>" for the
            delete to proceed. If empty or wrong, returns a preview and does
            nothing. The human confirms by providing the token explicitly.
    """
    expected = f"CONFIRM_DELETE_{ticket_id}"
    if confirmation_token != expected:
        return {
            "status": "preview",
            "ticket_id": ticket_id,
            "current_title": TICKETS.get(ticket_id, "<unknown>"),
            "message": (
                f"Delete not executed. To proceed, pass confirmation_token="
                f"'{expected}'. Ask the user to confirm before retrying."
            ),
        }
    if ticket_id not in TICKETS:
        return {"status": "error", "message": f"Ticket {ticket_id} not found."}
    title = TICKETS.pop(ticket_id)
    return {"status": "deleted", "ticket_id": ticket_id, "former_title": title}

guarded_agent = LlmAgent(
    name="guarded_ticket_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Manages tickets, with a confirmation gate on destructive operations.",
    instruction=(
        "You manage tickets. For delete_ticket: never call with "
        "confirmation_token on the first attempt. Always call it with an empty "
        "token first to preview, show the user the preview, and only call with "
        "the correct token if the user explicitly confirms."
    ),
    tools=[delete_ticket],
)

await chat(guarded_agent, "Delete ticket T-1001.")

USER: Delete ticket T-1001.



[tool_call] delete_ticket({'ticket_id': 'T-1001', 'confirmation_token': ''})
[tool_resp] {'status': 'preview', 'ticket_id': 'T-1001', 'current_title': "Laptop won't boot", 'message': "Delete not executed. To proceed, pass confirmation_token='CONFIRM_DELETE_T-1001'. Ask the user to confirm before retrying."}


[FINAL] guarded_ticket_agent: Ticket **T-1001** is titled **“Laptop won't boot.”**

Deletion has not been executed. Please explicitly confirm that you want to permanently delete this ticket.


### 🔍 What just happened?

The agent called `delete_ticket` without a token, got a **preview** back, and surfaced it to the user instead of destroying anything. The actual delete only happens if a second turn supplies the exact token — and that string-equality check lives in the *function*, where no clever prompt can talk its way around it. Carry this pattern to every tool with irreversible consequences.

### 🎯 Mini-task

Add a `wipe_all_tickets()` tool that returns a preview like *"this would delete N tickets"* and demands the token `"CONFIRM_WIPE_ALL"`. Verify the preview comes back before anything is deleted.

## Cleanup

Close the MCP toolset to shut down the ticket-server subprocess cleanly — otherwise every notebook restart leaks a background Python process. (`close()` is `async` like everything else in ADK, hence the `await`.)

In [13]:
await ticket_toolset.close()
print("✅ MCP subprocess closed.")

✅ MCP subprocess closed.


# Key Takeaways

The help desk got four kinds of tools in one day:

- **FunctionTool** — you write it; the docstring + type hints are the schema. The default choice.
- **OpenAPIToolset** — hand ADK an API's menu (the spec) and every endpoint becomes a tool.
- **McpToolset** — connect to a tool server that runs as its own program, in any language.
- **AgentTool** — a specialist agent, callable like a function; the parent stays in charge.
- Dangerous tools get their guard **in code**, not in the prompt. Blast radius decides the tier.

# Next up — Module 03

Every tool call today landed in the session's event history. M03 makes sessions the subject: how to persist them, how to put state in them, and the `user:` / `app:` / `temp:` prefixes that decide what survives.